In [0]:
from pyspark.sql import functions as F

VOL = "/Volumes/workspace/capstone_subhamdnath/raw"
GOLD = f"{VOL}/gold"

gold = (
    spark.read
    .format("delta")
    .load(f"{GOLD}/store_hour")
)

print("Gold rows:", gold.count())
print("Gold columns:")
print(gold.columns)

print("\nSensor status:")
gold.groupBy("sensor_ok").count().orderBy("sensor_ok").show()

Gold rows: 12960
Gold columns:
['store_id', 'city', 'format', 'trade_date', 'hour', 'is_weekend', 'footfall', 'bills', 'revenue', 'conversion_rate', 'sensor_ok']

Sensor status:
+---------+-----+
|sensor_ok|count|
+---------+-----+
|    false|  540|
|     true|12420|
+---------+-----+



In [0]:
# Q1: Conversion rate per store-hour

q1 = (
    gold
    .filter(F.col("sensor_ok"))
    .select(
        "store_id",
        "city",
        "format",
        "trade_date",
        "hour",
        "footfall",
        "bills",
        "conversion_rate"
    )
    .orderBy("store_id", "trade_date", "hour")
)

print("Usable store-hours:", q1.count())

display(q1.limit(30))

Usable store-hours: 12420


store_id,city,format,trade_date,hour,footfall,bills,conversion_rate
ST01,CITY00,Mall,2025-04-01,13,27,5,0.18518518518518517
ST01,CITY00,Mall,2025-04-01,14,22,2,0.09090909090909091
ST01,CITY00,Mall,2025-04-01,15,18,1,0.05555555555555555
ST01,CITY00,Mall,2025-04-01,16,17,1,0.058823529411764705
ST01,CITY00,Mall,2025-04-01,17,24,4,0.16666666666666666
ST01,CITY00,Mall,2025-04-01,18,49,9,0.1836734693877551
ST01,CITY00,Mall,2025-04-01,19,48,8,0.16666666666666666
ST01,CITY00,Mall,2025-04-01,20,48,8,0.16666666666666666
ST01,CITY00,Mall,2025-04-01,21,39,7,0.1794871794871795
ST01,CITY00,Mall,2025-04-02,10,24,4,0.16666666666666666


In [0]:
# Q2: Three worst hours across the entire chain

q2 = (
    gold
    .filter(F.col("sensor_ok"))
    .groupBy("hour")
    .agg(
        F.sum("footfall").alias("total_footfall"),
        F.sum("bills").alias("total_bills")
    )
    .withColumn(
        "conversion_rate",
        F.round(
            F.try_divide(
                F.col("total_bills").cast("double"),
                F.col("total_footfall").cast("double")
            ),
            4
        )
    )
    .orderBy("conversion_rate")
)

print("========== Q2: HOURLY CONVERSION ==========")

display(q2)

========== Q2: HOURLY CONVERSION ==========


hour,total_footfall,total_bills,conversion_rate
15,22765,1919,0.0843
16,20207,1712,0.0847
14,22516,1944,0.0863
19,68168,12710,0.1865
18,68417,12770,0.1866
10,33800,6312,0.1867
20,68094,12717,0.1868
17,33763,6310,0.1869
21,37839,7077,0.187
13,37882,7086,0.1871


In [0]:
# Q3: Does conversion drop on days with the highest footfall?

from pyspark.sql.window import Window

daily = (
    gold
    .filter(F.col("sensor_ok"))
    .groupBy("store_id", "trade_date")
    .agg(
        F.sum("footfall").alias("visitors"),
        F.sum("bills").alias("bills")
    )
)

window_spec = (
    Window
    .partitionBy("store_id")
    .orderBy("visitors")
)

deciles = (
    daily
    .withColumn("decile", F.ntile(10).over(window_spec))
)

q3 = (
    deciles
    .groupBy("decile")
    .agg(
        F.count("*").alias("store_days"),
        F.sum("visitors").alias("visitors"),
        F.sum("bills").alias("bills")
    )
    .withColumn(
        "conversion_rate",
        F.round(
            F.try_divide(
                F.col("bills").cast("double"),
                F.col("visitors").cast("double")
            ),
            4
        )
    )
    .orderBy("decile")
)

print("========== Q3: FOOTFALL DECILES ==========")

display(q3)

========== Q3: FOOTFALL DECILES ==========


decile,store_days,visitors,bills,conversion_rate
1,108,34538,6318,0.1829
2,108,38492,7096,0.1843
3,108,40463,7409,0.1831
4,108,41744,7671,0.1838
5,108,42929,7890,0.1838
6,108,44067,8112,0.1841
7,108,45558,8404,0.1845
8,108,57632,9164,0.159
9,108,66708,10424,0.1563
10,108,71102,11132,0.1566


In [0]:
# Export Gold table to CSV for Snowflake

EXPORT = f"{VOL}/export/gold_csv"

gold_export = (
    spark.read
    .format("delta")
    .load(f"{GOLD}/store_hour")
)

print("Rows to export:", gold_export.count())
print("Columns:", gold_export.columns)

(
    gold_export
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(EXPORT)
)

print("Gold CSV export completed.")
print("Export location:", EXPORT)

Rows to export: 12960
Columns: ['store_id', 'city', 'format', 'trade_date', 'hour', 'is_weekend', 'footfall', 'bills', 'revenue', 'conversion_rate', 'sensor_ok']
Gold CSV export completed.
Export location: /Volumes/workspace/capstone_subhamdnath/raw/export/gold_csv


In [0]:
# Check exported files

display(
    dbutils.fs.ls(EXPORT)
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/capstone_subhamdnath/raw/export/gold_csv/_SUCCESS,_SUCCESS,0,1790705993000
dbfs:/Volumes/workspace/capstone_subhamdnath/raw/export/gold_csv/_committed_5667454322649362390,_committed_5667454322649362390,113,1790705993000
dbfs:/Volumes/workspace/capstone_subhamdnath/raw/export/gold_csv/_started_5667454322649362390,_started_5667454322649362390,0,1790705993000
dbfs:/Volumes/workspace/capstone_subhamdnath/raw/export/gold_csv/part-00000-tid-5667454322649362390-6b92c79d-d4fe-4c40-8391-35e27b47b170-368-1-c000.csv,part-00000-tid-5667454322649362390-6b92c79d-d4fe-4c40-8391-35e27b47b170-368-1-c000.csv,972361,1790705993000


In [0]:
# Get the exact CSV file path

files = dbutils.fs.ls(EXPORT)

csv_file = [f.path for f in files if f.path.endswith(".csv")][0]

print(csv_file)

dbfs:/Volumes/workspace/capstone_subhamdnath/raw/export/gold_csv/part-00000-tid-5667454322649362390-6b92c79d-d4fe-4c40-8391-35e27b47b170-368-1-c000.csv
